# Coupled direct multi-horizon TSGAM forecasting

This notebook is a compact research walkthrough for the forecast-mode extension.

The point is not just to call the new API. The point is to make the statistical tradeoff visible:

- **Independent mode** fits one ordinary TSGAM regression per forecast horizon.
- **Coupled mode** fits the same horizon problems jointly, then penalizes jagged coefficient movement across the horizon axis.

For horizon $h$, the direct forecasting problem is

$$
X_t \longrightarrow y_{t+h}.
$$

With $T$ horizons, independent mode solves $T$ separate problems. Coupled mode solves one joint problem:

$$
\min_{\{\theta_h\}_{h=1}^T}
\sum_{h=1}^{T}
\mathcal{L}_h(\theta_h)
+ \sum_{h=1}^{T}\mathcal{R}_{\text{tsgam}}(\theta_h)
+ \lambda_{\text{couple}}
\sum_b \lVert D_1 \Theta_b \rVert_F^2.
$$

Here $\Theta_b$ is a coefficient block arranged as coefficients by horizon, and $D_1$ is the first-difference operator across adjacent horizons. Large $\lambda_{\text{couple}}$ says neighboring horizons should not learn wildly different coefficient values unless the data demands it.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from tsgam_estimator import (
    TsgamEstimatorConfig,
    TsgamForecastConfig,
    TsgamForecastCouplingConfig,
    TsgamForecastEstimator,
    TsgamLinearConfig,
    TsgamMultiPeriodicConfig,
    TsgamSolverConfig,
)

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind")
pd.options.display.float_format = "{:.3f}".format


## 1. Synthetic problem

We generate one hourly target series with three pieces:

$$
y_t = s_{\text{daily}}(t) + s_{\text{weekly}}(t) + g(x_{t-1}, x_{t-2}, \ldots) + \epsilon_t.
$$

The periodic pieces are deliberately simple. The exogenous driver is a standardized non-periodic random process. The target uses a known causal linear filter of that driver.

A direct forecast made at time $t$ cannot see $x_{t+1}, x_{t+2}, \ldots$. It only sees values available at forecast time, such as $x_t, x_{t-1}, x_{t-2}$. That is exactly the setup where different horizons can estimate related but noisy lag coefficients.


In [ ]:
HORIZON = 8
ORIGIN_LAGS = [-2, -1, 0]
MAX_ORIGIN_BACK = abs(min(ORIGIN_LAGS))
N_SAMPLES = 1_400
TRAIN_SAMPLES = 700
NOISE_SCALE = 0.10
RNG_SEED = 7
SELECTED_COUPLING_WEIGHT = 0.2
SWEEP_WEIGHTS = [0.0, 0.05, 0.2, 1.0, 5.0, 25.0]

index = pd.date_range("2024-01-01", periods=N_SAMPLES, freq="h")
train_end_time = index[TRAIN_SAMPLES]


In [ ]:
@dataclass(frozen=True)
class SyntheticForecastData:
    X: pd.DataFrame
    y: pd.Series
    components: pd.DataFrame
    target_filter: pd.Series


def make_periodic_components(t: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    daily = 1.25 * np.sin(2 * np.pi * t / 24 - 0.4)
    daily += 0.35 * np.cos(4 * np.pi * t / 24 + 0.2)
    weekly = 0.75 * np.sin(2 * np.pi * t / 168 + 0.7)
    return daily, weekly


def make_driver(rng: np.random.Generator, n: int) -> np.ndarray:
    raw = rng.normal(size=n)
    return (raw - raw.mean()) / raw.std()


def make_target_filter(max_target_lag: int) -> pd.Series:
    lags = np.arange(1, max_target_lag + 1)
    weights = 0.72 * np.exp(-0.23 * (lags - 1)) + 0.04 * np.sin(lags / 1.8)
    return pd.Series(weights, index=lags, name="target_time_filter")


def apply_target_time_filter(driver: np.ndarray, weights: pd.Series) -> np.ndarray:
    contribution = np.zeros_like(driver, dtype=float)
    for target_lag, weight in weights.items():
        contribution[target_lag:] += weight * driver[:-target_lag]
    return contribution


def make_synthetic_forecast_data() -> SyntheticForecastData:
    rng = np.random.default_rng(RNG_SEED)
    t = np.arange(N_SAMPLES)
    daily, weekly = make_periodic_components(t)
    driver = make_driver(rng, N_SAMPLES)

    max_target_lag = HORIZON + MAX_ORIGIN_BACK + 1
    target_filter = make_target_filter(max_target_lag)
    linear_exog = apply_target_time_filter(driver, target_filter)
    noise = rng.normal(scale=NOISE_SCALE, size=N_SAMPLES)
    y = daily + weekly + linear_exog + noise

    X = pd.DataFrame({"driver": driver}, index=index)
    components = pd.DataFrame(
        {
            "daily periodic": daily,
            "weekly periodic": weekly,
            "periodic total": daily + weekly,
            "linear exogenous contribution": linear_exog,
            "noise": noise,
            "observed target": y,
            "driver": driver,
        },
        index=index,
    )
    return SyntheticForecastData(X=X, y=pd.Series(y, index=index, name="target"), components=components, target_filter=target_filter)


data = make_synthetic_forecast_data()
display(data.target_filter.to_frame())


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
plot_window = data.components.iloc[:1_000]

axes[0].plot(plot_window.index, plot_window["driver"], color="tab:blue", linewidth=1.2)
axes[0].set_title("Origin-known exogenous driver")
axes[0].set_ylabel("x")

axes[1].plot(plot_window.index, plot_window["daily periodic"], label="daily", linewidth=1.1)
axes[1].plot(plot_window.index, plot_window["weekly periodic"], label="weekly", linewidth=1.1)
axes[1].plot(plot_window.index, plot_window["periodic total"], label="total", color="black", linewidth=1.0, alpha=0.75)
axes[1].set_title("Known periodic components")
axes[1].set_ylabel("value")
axes[1].legend(loc="upper right", ncols=3)

axes[2].plot(plot_window.index, plot_window["linear exogenous contribution"], color="tab:green", linewidth=1.2)
axes[2].set_title("Known linear contribution from lagged driver values")
axes[2].set_ylabel("value")

axes[3].plot(plot_window.index, plot_window["observed target"], color="tab:purple", linewidth=1.0)
axes[3].axvline(train_end_time, color="black", linestyle="--", linewidth=1.2, label="train/test split")
axes[3].set_title("Observed target used by the forecast estimator")
axes[3].set_ylabel("y")
axes[3].legend(loc="upper right")

for ax in axes:
    ax.axvline(train_end_time, color="black", linestyle="--", linewidth=0.9, alpha=0.55)

axes[-1].set_xlabel("time")
fig.tight_layout()


## 2. What each horizon sees

Forecast mode uses direct multi-horizon regression. For each horizon $h$, the model trains on rows like this:

$$
\bigl[x_t, x_{t-1}, x_{t-2}, \text{periodic basis at } t+h\bigr] \rightarrow y_{t+h}.
$$

In code, the forecaster shifts the feature frame's timestamp index to the target time, but keeps the feature values from the forecast origin. This lets the existing TSGAM timestamp and basis machinery build the target-time periodic basis while the exogenous columns remain values that were available when the forecast was made.


In [ ]:
origin_ix = TRAIN_SAMPLES + 48
origin_time = data.X.index[origin_ix]
alignment_rows = []
for h in range(1, HORIZON + 1):
    target_time = origin_time + pd.Timedelta(hours=h)
    alignment_rows.append(
        {
            "horizon": h,
            "forecast origin": origin_time,
            "target time": target_time,
            "origin driver x_t": data.X.loc[origin_time, "driver"],
            "older driver x_t-1": data.X.iloc[origin_ix - 1, 0],
            "older driver x_t-2": data.X.iloc[origin_ix - 2, 0],
            "actual y_t+h": data.y.loc[target_time],
        }
    )

display(pd.DataFrame(alignment_rows))


## 3. Fit the new API

The base TSGAM model has two parts:

- Fourier terms for daily and weekly periodicity;
- a linear exogenous term over lags available at the forecast origin: $[-2, -1, 0]$.

The independent forecaster fits one base estimator per horizon. The coupled forecaster fits one joint CVXPY problem and adds a first-difference penalty across horizon-indexed coefficient blocks.


In [ ]:
def make_base_config() -> TsgamEstimatorConfig:
    return TsgamEstimatorConfig(
        multi_periodic_config=TsgamMultiPeriodicConfig(
            num_harmonics=[3, 2],
            periods=[24, 168],
            reg_weight=1e-5,
        ),
        exog_config=[
            TsgamLinearConfig(
                lags=ORIGIN_LAGS,
                reg_weight=1e-6,
                diff_reg_weight=1e-6,
            )
        ],
        solver_config=TsgamSolverConfig(solver="CLARABEL", verbose=False),
    )


def fit_independent_forecaster() -> TsgamForecastEstimator:
    model = TsgamForecastEstimator(
        TsgamForecastConfig(
            horizon=HORIZON,
            base_config=make_base_config(),
            mode="independent",
        )
    )
    return model.fit(data.X.iloc[:TRAIN_SAMPLES], data.y.iloc[:TRAIN_SAMPLES].to_numpy())


def fit_coupled_forecaster(roughness_weight: float) -> TsgamForecastEstimator:
    model = TsgamForecastEstimator(
        TsgamForecastConfig(
            horizon=HORIZON,
            base_config=make_base_config(),
            mode="coupled",
            coupling_config=TsgamForecastCouplingConfig(
                roughness_weight=roughness_weight,
                roughness_order=1,
            ),
        )
    )
    return model.fit(data.X.iloc[:TRAIN_SAMPLES], data.y.iloc[:TRAIN_SAMPLES].to_numpy())


independent_model = fit_independent_forecaster()
coupled_model = fit_coupled_forecaster(SELECTED_COUPLING_WEIGHT)


## 4. Forecast accuracy

First we look at the actual forecasts. The coupled model should not radically change the fitted signal here. Its job is to stabilize horizon-specific coefficients, not to invent a different forecasting target.


In [ ]:
def prediction_origins() -> pd.DataFrame:
    # Include a little history before the test split so negative exogenous lags are available.
    stop = N_SAMPLES - HORIZON
    return data.X.iloc[TRAIN_SAMPLES - MAX_ORIGIN_BACK : stop]


def forecast_long_frame(model: TsgamForecastEstimator, model_name: str) -> pd.DataFrame:
    predictions = model.predict(prediction_origins())
    rows = []
    for h in range(1, HORIZON + 1):
        target_times = predictions.index + pd.to_timedelta(h, unit="h")
        actual = data.y.reindex(target_times).to_numpy()
        frame = pd.DataFrame(
            {
                "model": model_name,
                "origin_time": predictions.index,
                "target_time": target_times,
                "horizon": h,
                "prediction": predictions[f"horizon_{h}"].to_numpy(),
                "actual": actual,
            }
        )
        rows.append(frame)
    result = pd.concat(rows, ignore_index=True)
    result = result[result["origin_time"] >= train_end_time].copy()
    result["error"] = result["prediction"] - result["actual"]
    result["absolute_error"] = result["error"].abs()
    return result


independent_forecasts = forecast_long_frame(independent_model, "independent")
coupled_forecasts = forecast_long_frame(coupled_model, f"coupled, weight={SELECTED_COUPLING_WEIGHT:g}")
forecast_results = pd.concat([independent_forecasts, coupled_forecasts], ignore_index=True)

metrics_by_horizon = (
    forecast_results.groupby(["model", "horizon"])
    .apply(
        lambda frame: pd.Series(
            {
                "RMSE": np.sqrt(np.mean(frame["error"] ** 2)),
                "MAE": np.mean(frame["absolute_error"]),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

display(metrics_by_horizon.pivot(index="horizon", columns="model", values="RMSE"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
sns.lineplot(data=metrics_by_horizon, x="horizon", y="RMSE", hue="model", marker="o", ax=axes[0])
sns.lineplot(data=metrics_by_horizon, x="horizon", y="MAE", hue="model", marker="o", ax=axes[1])
axes[0].set_title("RMSE by forecast horizon")
axes[1].set_title("MAE by forecast horizon")
for ax in axes:
    ax.set_xlabel("forecast horizon")
    ax.set_ylabel("error")
fig.tight_layout()


In [ ]:
selected_origin = train_end_time + pd.Timedelta(days=4)
path_rows = []
for model_name, frame in [
    ("actual", independent_forecasts),
    ("independent", independent_forecasts),
    (f"coupled, weight={SELECTED_COUPLING_WEIGHT:g}", coupled_forecasts),
]:
    origin_frame = frame[frame["origin_time"] == selected_origin]
    if model_name == "actual":
        values = origin_frame["actual"].to_numpy()
    else:
        values = origin_frame["prediction"].to_numpy()
    for h, value in zip(origin_frame["horizon"], values, strict=True):
        path_rows.append(
            {
                "series": model_name,
                "horizon": h,
                "target_time": selected_origin + pd.Timedelta(hours=int(h)),
                "value": value,
            }
        )

path_frame = pd.DataFrame(path_rows)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(data=path_frame, x="target_time", y="value", hue="series", marker="o", ax=ax)
ax.set_title("One forecast origin expanded into an 8-hour forecast path")
ax.set_xlabel("target time")
ax.set_ylabel("target / forecast")
fig.tight_layout()


In [ ]:
plot_horizons = [1, 4, 8]
fig, axes = plt.subplots(len(plot_horizons), 1, figsize=(13, 8), sharex=True)
for ax, h in zip(axes, plot_horizons, strict=True):
    actual_frame = independent_forecasts[independent_forecasts["horizon"] == h].iloc[:220]
    independent_frame = independent_forecasts[independent_forecasts["horizon"] == h].iloc[:220]
    coupled_frame = coupled_forecasts[coupled_forecasts["horizon"] == h].iloc[:220]
    ax.plot(actual_frame["target_time"], actual_frame["actual"], color="black", linewidth=1.0, label="actual")
    ax.plot(independent_frame["target_time"], independent_frame["prediction"], color="tab:red", linestyle="--", linewidth=1.0, label="independent")
    ax.plot(coupled_frame["target_time"], coupled_frame["prediction"], color="tab:blue", linewidth=1.0, label="coupled")
    ax.set_title(f"Horizon {h}: forecast vs actual")
    ax.set_ylabel("value")
axes[-1].set_xlabel("target time")
axes[0].legend(loc="upper right", ncols=3)
fig.tight_layout()


## 5. What the coupling changes

The cleanest way to read the coefficient plot is in **filter lag** coordinates.

The synthetic target contains a known linear filter: `target_filter[k]` is the planted coefficient multiplying `x_{target_time-k}`. A forecast made at time `t` for horizon `h` predicts `y_{t+h}`. If the model uses a visible feature `x_{t+ell}`, then that feature is `k = h - ell` steps behind the target.

So each learned forecast coefficient maps back to one planted filter tap:

$$
\text{learned coefficient for horizon } h \text{ and origin lag } \ell
\quad\longleftrightarrow\quad
\text{target\_filter}[h - \ell].
$$

The plot below uses that mapping. The black line is the actual filter used to generate the data. The fitted lines show how well the independent and coupled models recover those filter taps. When several horizon/lag coefficients map to the same filter lag, the fitted line uses their average.


In [ ]:
def true_filter_coefficients() -> pd.DataFrame:
    rows = []
    for h in range(1, HORIZON + 1):
        for lag in ORIGIN_LAGS:
            filter_lag = h - lag
            rows.append(
                {
                    "model": "true filter",
                    "horizon": h,
                    "origin lag": lag,
                    "filter lag": filter_lag,
                    "coefficient": data.target_filter.loc[filter_lag],
                }
            )
    return pd.DataFrame(rows)


def fitted_linear_coefficients(model: TsgamForecastEstimator, model_name: str) -> pd.DataFrame:
    rows = []
    if model.config.mode == "independent":
        for h in model.horizons_:
            coef = model.forecast_estimators_[h].variables_["exog_coef_0"].value[0, :]
            for lag, value in zip(ORIGIN_LAGS, coef, strict=True):
                rows.append({"model": model_name, "horizon": h, "origin lag": lag, "filter lag": h - lag, "coefficient": value})
    else:
        horizon_vars = model.variables_["exog_coef_0"]
        for h, variable in zip(model.horizons_, horizon_vars, strict=True):
            coef = variable.value[0, :]
            for lag, value in zip(ORIGIN_LAGS, coef, strict=True):
                rows.append({"model": model_name, "horizon": h, "origin lag": lag, "filter lag": h - lag, "coefficient": value})
    return pd.DataFrame(rows)


coef_frame = pd.concat(
    [
        true_filter_coefficients(),
        fitted_linear_coefficients(independent_model, "independent"),
        fitted_linear_coefficients(coupled_model, f"coupled, weight={SELECTED_COUPLING_WEIGHT:g}"),
    ],
    ignore_index=True,
)

display(coef_frame.head())


In [ ]:
filter_plot = (
    coef_frame
    .groupby(["model", "filter lag"], as_index=False)["coefficient"]
    .mean()
)

model_styles = {
    "independent": {"color": "tab:red", "marker": "x", "linestyle": "--", "offset": -0.06},
    f"coupled, weight={SELECTED_COUPLING_WEIGHT:g}": {"color": "tab:blue", "marker": "o", "linestyle": "-", "offset": 0.06},
}

fig, ax = plt.subplots(figsize=(9.5, 5.0))
true_frame = filter_plot[filter_plot["model"] == "true filter"]
ax.plot(
    true_frame["filter lag"],
    true_frame["coefficient"],
    color="black",
    marker="o",
    linewidth=2.4,
    label="true filter",
)

for model_name, style in model_styles.items():
    point_frame = coef_frame[coef_frame["model"] == model_name]
    line_frame = filter_plot[filter_plot["model"] == model_name]
    ax.scatter(
        point_frame["filter lag"] + style["offset"],
        point_frame["coefficient"],
        color=style["color"],
        marker=style["marker"],
        alpha=0.45,
        s=42,
        linewidths=1.2,
    )
    ax.plot(
        line_frame["filter lag"],
        line_frame["coefficient"],
        color=style["color"],
        linestyle=style["linestyle"],
        linewidth=1.9,
        label=f"{model_name} mean",
    )

ax.axhline(0, color="black", linewidth=0.7, alpha=0.35)
ax.set_title("Recovering the planted linear filter")
ax.set_xlabel("filter lag k in target_filter[k]")
ax.set_ylabel("linear coefficient")
ax.legend(loc="best")
fig.tight_layout()


In [ ]:
def coefficient_matrix(coef_data: pd.DataFrame, model_name: str) -> np.ndarray:
    pivot = (
        coef_data[coef_data["model"] == model_name]
        .pivot(index="origin lag", columns="horizon", values="coefficient")
        .loc[ORIGIN_LAGS]
    )
    return pivot.to_numpy()


def coefficient_summary(coef_data: pd.DataFrame, model_name: str) -> dict[str, float | str]:
    matrix = coefficient_matrix(coef_data, model_name)
    true_filter = coefficient_matrix(coef_data, "true filter")
    return {
        "model": model_name,
        "first_difference_roughness": float(np.sum(np.diff(matrix, axis=1) ** 2)),
        "coefficient_RMSE_vs_true_filter": float(np.sqrt(np.mean((matrix - true_filter) ** 2))),
    }

coefficient_diagnostics = pd.DataFrame(
    [
        coefficient_summary(coef_frame, "independent"),
        coefficient_summary(coef_frame, f"coupled, weight={SELECTED_COUPLING_WEIGHT:g}"),
    ]
)
display(coefficient_diagnostics)


## 6. Coupling strength sweep

The coupling weight controls a bias-variance tradeoff.

- At zero or tiny weight, the horizons can wiggle independently.
- At moderate weight, neighboring horizons borrow strength.
- At very large weight, the coefficients become nearly flat across horizons and may underfit real horizon variation.


In [ ]:
sweep_rows = []
sweep_coef_frames = []
for weight in SWEEP_WEIGHTS:
    model = fit_coupled_forecaster(weight)
    model_name = f"coupled weight={weight:g}"
    model_forecasts = forecast_long_frame(model, model_name)
    model_coefs = pd.concat(
        [true_filter_coefficients(), fitted_linear_coefficients(model, model_name)],
        ignore_index=True,
    )
    coef_matrix_value = coefficient_matrix(model_coefs, model_name)
    true_filter_matrix = coefficient_matrix(model_coefs, "true filter")
    roughness = np.sum(np.diff(coef_matrix_value, axis=1) ** 2)
    coef_rmse = np.sqrt(np.mean((coef_matrix_value - true_filter_matrix) ** 2))
    mean_rmse = np.sqrt(np.mean(model_forecasts["error"] ** 2))
    mean_mae = np.mean(model_forecasts["absolute_error"])
    sweep_rows.append(
        {
            "roughness_weight": weight,
            "coefficient_roughness": roughness,
            "coefficient_RMSE_vs_true_filter": coef_rmse,
            "forecast_RMSE": mean_rmse,
            "forecast_MAE": mean_mae,
        }
    )
    model_coefs["roughness_weight"] = weight
    sweep_coef_frames.append(model_coefs[model_coefs["model"] == model_name])

sweep = pd.DataFrame(sweep_rows)
independent_summary = coefficient_summary(coef_frame, "independent")
independent_mean_rmse = np.sqrt(np.mean(independent_forecasts["error"] ** 2))
independent_summary["forecast_RMSE"] = independent_mean_rmse

display(sweep)
display(pd.DataFrame([independent_summary]))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_specs = [
    ("coefficient_roughness", "coefficient roughness"),
    ("coefficient_RMSE_vs_true_filter", "coefficient RMSE vs true filter"),
    ("forecast_RMSE", "test forecast RMSE"),
]
for ax, (column, title) in zip(axes, plot_specs, strict=True):
    sns.lineplot(data=sweep, x="roughness_weight", y=column, marker="o", ax=ax)
    if column in independent_summary:
        ax.axhline(independent_summary[column], color="tab:red", linestyle="--", linewidth=1.0, label="independent")
        ax.legend(loc="best")
    ax.set_xscale("symlog", linthresh=0.05)
    ax.set_title(title)
    ax.set_xlabel("coupling roughness weight")
fig.tight_layout()


## 7. Calibration and residuals

The coefficient plots show what coupling changes internally. The next plots check whether that change leaks into forecast behavior.


In [ ]:
calibration_horizons = [1, 4, 8]
calibration_data = forecast_results[forecast_results["horizon"].isin(calibration_horizons)].copy()

fig, axes = plt.subplots(1, len(calibration_horizons), figsize=(14, 4), sharex=True, sharey=True)
for ax, h in zip(axes, calibration_horizons, strict=True):
    frame = calibration_data[calibration_data["horizon"] == h]
    sns.scatterplot(
        data=frame,
        x="actual",
        y="prediction",
        hue="model",
        s=18,
        alpha=0.55,
        ax=ax,
        legend=h == calibration_horizons[0],
    )
    low = min(frame["actual"].min(), frame["prediction"].min())
    high = max(frame["actual"].max(), frame["prediction"].max())
    ax.plot([low, high], [low, high], color="black", linewidth=1.0, alpha=0.75)
    ax.set_title(f"horizon {h}")
    ax.set_xlabel("actual")
axes[0].set_ylabel("prediction")
fig.suptitle("Calibration: predictions should lie near the diagonal", y=1.03, fontsize=14)
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.boxplot(data=forecast_results, x="horizon", y="error", hue="model", ax=ax, showfliers=False)
ax.axhline(0, color="black", linewidth=0.9, alpha=0.7)
ax.set_title("Residual distribution by horizon")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("prediction - actual")
fig.tight_layout()


## 8. Takeaways

This notebook is set up so the forecast target is simple and mostly recoverable. The interesting part is not whether the model can find a daily cycle. It can. The interesting part is the horizon axis.

| Mode | What it solves | What to inspect |
| --- | --- | --- |
| Independent | One normal TSGAM fit per horizon | Forecast accuracy and coefficient jaggedness |
| Coupled | One joint problem with horizon roughness | Accuracy, coefficient smoothness, and oversmoothing |

The practical story is:

1. Independent mode is the easiest baseline and should remain the first comparison.
2. Coupled mode is useful when horizon-specific estimates are noisy but expected to vary smoothly.
3. The coupling weight is a tuning parameter. It can stabilize coefficients, but too much coupling can flatten real horizon structure.
